# IDEF 2022/2023 — análise em SQLite

Quanto gastam as famílias portuguesas e como se reparte essa despesa? Este notebook retoma quatro perguntas da análise exploratória. O Python carrega os CSV preparados no notebook 1 e apresenta os resultados. As seleções, comparações e contas são feitas em SQL.

Os valores referem-se à despesa anual média por agregado. As regiões seguem a delimitação NUTS II de 2013 e os grupos de rendimento usam o rendimento equivalente de 2021, ajustado à dimensão e composição do agregado.

## Carregar os dados

A base SQLite é criada em memória e volta a ser preenchida a cada execução. Os códigos COICOP são importados como texto para preservar os zeros à esquerda. O nível 0 corresponde ao total, o nível 1 às categorias principais e os níveis seguintes aos seus desdobramentos.

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
root = cwd.parent if cwd.name == "notebooks" else cwd
data = root / "data"

PROC = data / "processed"


base = sqlite3.connect(":memory:")
for tabela in ("despesa_por_regiao", "despesa_por_quintil"):
    csv = pd.read_csv(PROC / f"{tabela}.csv", dtype={"coicop": str})
    csv.to_sql(tabela, base, index=False, if_exists="replace")

## 1. Que categorias ocupam a maior parte do orçamento?

`LIMIT` apresenta as três primeiras. categorias da seleção.

In [2]:
categorias = pd.read_sql_query("""
SELECT coicop, descricao AS categoria, despesa_eur AS euros, peso_pct
FROM despesa_por_regiao
WHERE grupo = 'Portugal' AND nivel = 1
ORDER BY peso_pct DESC, coicop
LIMIT 3
""", base)
categorias

,coicop,categoria,euros,peso_pct
0,04,"Habitação, água, eletricidade, gás e outros co...",9390.0,39.3
1,01,Produtos alimentares e bebidas não alcoólicas,3091.0,12.9
2,07,Transportes,2888.0,12.1


Habitação, alimentação e transportes somam 64,3% da despesa publicada. Só a habitação representa 39,3%, incluindo a renda estimada pelas famílias que vivem em casa própria ou cedida.

## 2. Quanto pesa a habitação em cada região se retirarmos a renda subjetiva?

A consulta agrupa as linhas por região com `GROUP BY`. Para cada região, `MAX(...) FILTER (WHERE ...)` coloca os valores de habitação (`04`), renda subjetiva (`04.2`) e despesa total (`TOTAL`) em colunas distintas. Depois subtrai a renda subjetiva à habitação e ao total. As restantes despesas não monetárias, como o autoconsumo, continuam incluídas. Portugal aparece como referência e o agregado Continente fica fora da comparação.

In [3]:
habitacao = pd.read_sql_query("""
WITH valores AS (
    SELECT
        grupo AS regiao,
        MAX(peso_pct) FILTER (WHERE coicop = '04') AS peso_publicado_pct,
        MAX(despesa_eur) FILTER (WHERE coicop = '04') AS habitacao,
        MAX(despesa_eur) FILTER (WHERE coicop = '04.2') AS renda_subjetiva,
        MAX(despesa_eur) FILTER (WHERE coicop = 'TOTAL') AS total
    FROM despesa_por_regiao
    WHERE coicop IN ('04', '04.2', 'TOTAL')
      AND grupo <> 'Continente'
    GROUP BY grupo
)
SELECT
    regiao,
    peso_publicado_pct,
    ROUND(
        100.0 * (habitacao - renda_subjetiva)
        / NULLIF(total - renda_subjetiva, 0),
        1
    ) AS sem_renda_subjetiva_pct
FROM valores
ORDER BY (regiao = 'Portugal') DESC, peso_publicado_pct DESC, regiao
""", base)
habitacao

,regiao,peso_publicado_pct,sem_renda_subjetiva_pct
0,Portugal,39.3,16.4
1,A.M. Lisboa,43.8,18.0
2,R.A. Madeira,39.7,15.2
3,R.A. Açores,39.2,16.7
4,Algarve,38.8,15.8
5,Centro,37.9,15.1
6,Norte,36.9,16.3
7,Alentejo,33.4,14.6


Lisboa tem o maior peso nos dois cenários. A diferença face ao Alentejo cai de 10,4 para 3,4 pontos percentuais quando se retira a renda subjetiva. O notebook 2 mostra também o cenário de renda imputada, com valores da publicação do INE.

## 3. Como muda o peso da alimentação e da habitação com o rendimento?

Um quintil é um dos cinco grupos em que as famílias foram divididas depois de ordenadas pelo rendimento ajustado. O primeiro reúne as famílias com menor rendimento ajustado. `CASE` seleciona a categoria e `MAX` coloca o seu valor numa coluna por quintil. Há um único peso por categoria e quintil.

In [4]:
pesos = pd.read_sql_query("""
SELECT
    grupo AS quintil,
    MAX(CASE WHEN coicop = '01' THEN peso_pct END) AS alimentacao_pct,
    MAX(CASE WHEN coicop = '04' THEN peso_pct END) AS habitacao_pct
FROM despesa_por_quintil
WHERE coicop IN ('01', '04') AND grupo <> 'Total'
GROUP BY grupo
ORDER BY grupo
""", base)
pesos

,quintil,alimentacao_pct,habitacao_pct
0,1º quintil,15.8,41.4
1,2º quintil,15.1,39.8
2,3º quintil,14.6,37.6
3,4º quintil,13.0,37.6
4,5º quintil,9.8,40.1


A alimentação perde peso em cada passagem para o quintil seguinte. A habitação desce de 41,4% no primeiro quintil para 37,6% no terceiro e no quarto, e volta a subir para 40,1% no quinto.

## 4. Quanto gasta o quinto quintil em relação ao primeiro?

A consulta junta os dois quintis pelo código da categoria. Divide a despesa do quinto quintil pela do primeiro, em euros. Um rácio de 2 significa que o quinto quintil gasta, em média, o dobro. Incluímos também a linha do total para dar contexto aos rácios de cada categoria.

In [5]:
racios = pd.read_sql_query("""
SELECT
    primeiro.coicop,
    primeiro.descricao AS categoria,
    primeiro.despesa_eur AS primeiro_eur,
    quinto.despesa_eur AS quinto_eur,
    ROUND(1.0 * quinto.despesa_eur / NULLIF(primeiro.despesa_eur, 0), 1) AS racio
FROM despesa_por_quintil AS primeiro
JOIN despesa_por_quintil AS quinto
    ON quinto.coicop = primeiro.coicop AND quinto.grupo = '5º quintil'
WHERE primeiro.grupo = '1º quintil' AND primeiro.nivel IN (0, 1)
ORDER BY primeiro.nivel, racio DESC, primeiro.coicop
""", base)
racios

,coicop,categoria,primeiro_eur,quinto_eur,racio
0,TOTAL,Despesa total anual média por agregado,16294.0,34994.0,2.1
1,09,"Lazer, recreação desporto e cultura",335.0,1560.0,4.7
2,10,Serviços de educação,149.0,706.0,4.7
3,05,"Acessórios para o lar, equipamento doméstico e...",498.0,2172.0,4.4
4,12,Seguros e serviços financeiros,417.0,1381.0,3.3
5,11,Restaurantes e serviços de alojamento,1243.0,3256.0,2.6
6,07,Transportes,1841.0,4032.0,2.2
7,03,Vestuário e calçado,266.0,556.0,2.1
8,04,"Habitação, água, eletricidade, gás e outros co...",6754.0,14024.0,2.1
9,13,"Cuidados pessoais, proteção social e bens dive...",586.0,1101.0,1.9


A despesa total do quinto quintil é 2,1 vezes a do primeiro, incluindo a renda subjetiva. Na alimentação, o rácio é de 1,3. Isso explica como o quinto quintil pode gastar mais euros em alimentação e dedicar-lhe uma fatia menor do orçamento.

Fonte: INE, *Inquérito às Despesas das Famílias 2022/2023*, quadros Q.2.1.a/b e Q.2.11.a/b. O notebook 2 e o README descrevem as limitações das comparações.